In [1]:
from pynq import Overlay,allocate
import numpy as np


In [2]:
# test data
test_data = np.array(range(0, 10000), dtype=np.uint32) # List from 0 to 9999
test_data_list = list(range(0, 10000))

In [3]:
# setup axi stream overlay
ol_axis = Overlay("/home/xilinx/jupyter_notebooks/Homeworks/HW4/divby13_stream.bit")
dma = ol_axis.axi_dma
dma_send = ol_axis.axi_dma.sendchannel
dma_recv = ol_axis.axi_dma.recvchannel
divby13_stream_ip = ol_axis.divby13_stream_0
CR = 0x00
divby13_stream_ip.write(CR, 0x01)

# HW Implementation of divby13_stream
def divby13_stream_hw(data):
    
    input_buffer = allocate(shape=(len(data),), dtype=np.uint32)
    output_buffer = allocate(shape=(len(data),), dtype=np.uint32)
    
    np.copyto(input_buffer, data)
    
    dma_send.transfer(input_buffer)
    dma_recv.transfer(output_buffer)

    
    result = output_buffer.copy()
    
    del input_buffer, output_buffer
    
    return result


In [4]:
divby13_stream_ip.register_map

RegisterMap {
  CTRL = Register(AP_START=1, AP_DONE=0, AP_IDLE=0, AP_READY=0, RESERVED_1=0, AUTO_RESTART=0, RESERVED_2=0, INTERRUPT=0, RESERVED_3=0),
  GIER = Register(Enable=0, RESERVED=0),
  IP_IER = Register(CHAN0_INT_EN=0, CHAN1_INT_EN=0, RESERVED_0=0),
  IP_ISR = Register(CHAN0_INT_ST=0, CHAN1_INT_ST=0, RESERVED_0=0)
}

In [5]:
# Test and time AXIStream Implementation with %time
%time divby13_stream_hw(test_data)

CPU times: user 3.23 ms, sys: 0 ns, total: 3.23 ms
Wall time: 2.92 ms


PynqBuffer([1, 0, 0, ..., 1, 0, 0], dtype=uint32)

In [6]:
# setup axilite overlay
ol_axilite = Overlay('/home/xilinx/jupyter_notebooks/Homeworks/HW3/divby13.bit')
divby13_ip = ol_axilite.divby13_0
divby13_ip.register_map

# HW Implementation of divby13
def divby13_axilite_hw(num):
    divby13_ip.write(0x10, num)
    return divby13_ip.read(0x18)

In [7]:
# Test and time AXILITE Implementation with %time
%time axiliteS_hw = [divby13_axilite_hw(num) for num in test_data_list]

CPU times: user 209 ms, sys: 0 ns, total: 209 ms
Wall time: 206 ms


In [10]:
# SW Implementation of divby13
def divby13_sw(num):
    return 1 if num % 13 == 0 else 0

In [12]:
# Test and time SW Implementation with %time and list
%time sw_results = [divby13_sw(num) for num in test_data_list]

CPU times: user 12.8 ms, sys: 0 ns, total: 12.8 ms
Wall time: 12.2 ms


In [13]:
# Test and time SW Implementation with %time and array 
%time sw_results = [divby13_sw(num) for num in test_data]

CPU times: user 228 ms, sys: 3.26 ms, total: 231 ms
Wall time: 228 ms


In [ ]:
S